# Regressão Logística com Normalização Standard e Balanceamento Undersampling

In [1]:
from pandas import read_csv

# Carregando os dados
arquivo = 'dados_dataset/dados_limpos.csv'
dados = read_csv(arquivo)
print("dataset carregado")

dataset carregado


## Balanceamento Undersampling

In [2]:
from sklearn import preprocessing
import pandas as pd

# 1. Transformar a coluna Attrition (0 e 1) em um formato que o LabelEncoder reconhece 
label_encoder = preprocessing.LabelEncoder()
dados['Attrition'] = label_encoder.fit_transform(dados['Attrition'])

# 2. Contagem das classes
count_class_0, count_class_1 = dados.Attrition.value_counts()

# 3. Separando as classes
target_class_0 = dados[dados['Attrition'] == 0]
target_class_1 = dados[dados['Attrition'] == 1]

# 4. Realizando o undersampling na classe majoritária
dados_class_0_under = target_class_0.sample(count_class_1)

# 5. Combinando as classes 
dados_under = pd.concat([dados_class_0_under, target_class_1], axis=0)

# Verificação da nova distribuição
print('Random undersampling:')
print(dados_under.Attrition.value_counts())

Random undersampling:
Attrition
0    237
1    237
Name: count, dtype: int64


## Normalização Standard

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Separando os dados 
X = dados_under.drop('Attrition', axis=1).values
Y = dados_under['Attrition'].values

# 2. Divisão 
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 3. Padronização 
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Tamanho do X_train após balanceamento:", X_train_scaled.shape)
print("Dados de Treino Padronizados (5 primeiras linhas):\n", X_train_scaled[0:5,:])

Tamanho do X_train após balanceamento: (379, 44)
Dados de Treino Padronizados (5 primeiras linhas):
 [[ 2.27513042 -1.40071827  2.25642657  0.15847399 -1.41283833 -0.51821278
  -0.85913213  2.00260013  1.2431196   1.90605639  1.07649955  0.08210834
  -0.82472775 -0.42073582  0.32829843  0.28555527  2.920844    0.11355173
  -0.97647672 -0.90845704 -1.07510215 -0.71258879 -1.05639689 -0.48842818
   0.61904093 -1.27035157  1.34439853 -0.86402765  2.72437557 -0.61089148
  -0.22973415 -0.36241314  0.7567127  -0.14684462 -0.54991408  3.84599359
  -0.265747   -0.14684462 -0.50905423 -0.558049   -0.31897364 -0.83184391
  -0.80933419 -0.78279369]
 [-0.46291735 -1.03474856  0.9311177   2.09594634  1.2292623   1.10251422
   1.7881124  -0.840042   -1.41369804 -0.64768015  0.99259832 -1.09962814
   0.5248915  -0.42073582  0.32829843 -0.80763108 -1.0559248   0.88205359
   0.3891494  -0.90845704 -1.07510215 -0.71258879 -1.05639689 -0.48842818
   0.61904093 -1.27035157 -0.74382706 -0.86402765 -0.36705

## Regressão Logística

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Instancia o modelo
modelo = LogisticRegression()

# Treina usando os dados padronizados
modelo.fit(X_train_scaled, y_train)

# Faz a previsão no conjunto de teste 
X_test_scaled = scaler.transform(X_test)
y_pred = modelo.predict(X_test_scaled)

# Avalia
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.77      0.70      0.73        47
           1       0.73      0.79      0.76        48

    accuracy                           0.75        95
   macro avg       0.75      0.75      0.75        95
weighted avg       0.75      0.75      0.75        95



In [5]:
import csv
import os
import time
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# tempo de predição
inicio = time.time()
y_pred = modelo.predict(X_test_scaled)
fim = time.time()

# Calcular as métricas
acuracia = accuracy_score(y_test, y_pred)
precisao = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
especificidade = tn / (tn + fp)

tempo = fim - inicio

# SALVANDO RESULTADOS

filename = 'resultados_experimentos.csv'

# Nomes das colunas
fields = ['Modelo', 'Descrição', 'Acurácia', 'Precisão', 'Recall', 'Specificity', 'F1 Score', 'Tempo em Segundo'] 
    
rows = [[
    'Regressão Logística', 
    'normalizacao standard, balanceamento under', 
    f"{acuracia:.8f}", 
    f"{precisao:.8f}", 
    f"{recall:.8f}", 
    f"{especificidade:.8f}", 
    f"{f1:.8f}", 
    f"{tempo:.8f}"
]]


file_exists = os.path.isfile(filename)

with open(filename, 'a', newline='', encoding='utf-8') as f:
    
    write = csv.writer(f)
    
    if not file_exists:
        write.writerow(fields) 
        
    # Escreve a linha com os resultados do modelo
    write.writerows(rows)
    
    f.close()

print(f"Resultados salvos com sucesso em {filename}")

Resultados salvos com sucesso em resultados_experimentos.csv
